# 08 — Indicators with uncertainty (Book 2, Chapter 8)

In [1]:
import time, numpy as np, pandas as pd, yaml
from pathlib import Path
from Bio import Entrez
ROOT = Path("..").resolve() if Path("../config.yml").exists() else Path(".").resolve()
FIN, LOGS = ROOT/"data/final", ROOT/"logs"
cfg = yaml.safe_load(open(ROOT/"config.yml")); Entrez.email = cfg["email"]
pc = pd.read_parquet(FIN/"paper_country.parquet"); papers = pd.read_parquet(FIN/"papers.parquet")

## Full and fractional counts, 2015–2024

In [2]:
w = pc[(pc.year >= 2015) & (pc.year <= 2024)]
cnt = pd.DataFrame({"full": w.groupby("country")["pmid"].nunique(), "fractional": w.groupby("country")["weight"].sum().round(1)})
cnt["rank_full"] = cnt.full.rank(ascending=False, method="min").astype(int)
cnt["rank_frac"] = cnt.fractional.rank(ascending=False, method="min").astype(int)
cnt = cnt.sort_values("fractional", ascending=False); print(cnt.head(15).to_string())

         full  fractional  rank_full  rank_frac
country                                        
USA      1746      1037.5          1          1
GBR       743       311.2          2          2
IND       347       249.6          4          3
AUS       419       230.6          3          4
CHN       229       150.7          9          5
DEU       317       134.8          5          6
FRA       292       118.1          6          7
BRA       186       115.6         12          8
NLD       281       110.9          7          9
CHE       243        92.2          8         10
JPN       175        91.6         13         11
GHA       191        78.3         11         12
THA       162        76.7         15         13
IRN        85        74.6         27         14
KEN       199        70.8         10         15


## Activity index against all malaria research
AI = (country's share of malaria-vaccine papers) / (country's share of all malaria papers).
The baseline comes from PubMed affiliation searches, which are rough; the code logs the queries.

In [3]:
NAMES = {"USA":"USA OR United States","GBR":"United Kingdom OR UK OR England OR Scotland","AUS":"Australia","FRA":"France",
         "DEU":"Germany","KEN":"Kenya","MLI":"Mali","BFA":"Burkina Faso","GHA":"Ghana","TZA":"Tanzania","NLD":"Netherlands",
         "IND":"India","CHE":"Switzerland","DNK":"Denmark","BRA":"Brazil","THA":"Thailand","CHN":"China","GAB":"Gabon"}
def n_pub(q):
    for _ in range(3):
        try: return int(Entrez.read(Entrez.esearch(db="pubmed", term=q, retmax=0))["Count"])
        except Exception: time.sleep(2)
base, vac, log = {}, {}, []
for iso, nm in NAMES.items():
    aff = "(" + " OR ".join(f'"{x.strip()}"[ad]' for x in nm.split(" OR ")) + ")"
    base[iso] = n_pub(f"malaria[mh] AND {aff} AND 2015:2024[dp]")
    vac[iso] = n_pub(f"({cfg['pubmed']['term']}) AND {aff} AND 2015:2024[dp]"); time.sleep(0.4)
    log.append((iso, aff))
tot_base = n_pub("malaria[mh] AND 2015:2024[dp]"); tot_vac = n_pub(f"({cfg['pubmed']['term']}) AND 2015:2024[dp]")
ai = pd.DataFrame({"vaccine_papers": vac, "malaria_papers": base})
ai["AI"] = ((ai.vaccine_papers / tot_vac) / (ai.malaria_papers / tot_base)).round(2)
print("totals:", tot_vac, "vaccine;", tot_base, "malaria"); ai = ai.sort_values("AI", ascending=False); print(ai.to_string())
ai.to_csv(LOGS/"activity_index.csv")

totals: 4443 vaccine; 23513 malaria
     vaccine_papers  malaria_papers    AI
DNK             153             442  1.83
NLD             283             964  1.55
DEU             321            1225  1.39
MLI             115             459  1.33
USA            1574            6567  1.27
AUS             429            1875  1.21
GAB              62             273  1.20
GHA             192             897  1.13
IND             352            1751  1.06
BRA             183            1014  0.96
FRA             294            1654  0.94
KEN             203            1166  0.92
CHN             231            1392  0.88
GBR             783            5014  0.83
CHE             244            1561  0.83
BFA              92             588  0.83
TZA             127             866  0.78
THA             168            1388  0.64


## Bootstrap interval for a share
Share of papers (with a known first-author country) whose first author is in sub-Saharan Africa.

In [4]:
rng = np.random.default_rng(2026)
def boot_share(x, B=2000):
    x = np.asarray(x, float); est = x.mean()
    bs = rng.choice(x, (B, len(x)), replace=True).mean(1)
    return est, *np.percentile(bs, [2.5, 97.5])
rows = []
for lo, hi in [(2000, 2009), (2010, 2019), (2020, 2026)]:
    g = papers[(papers.year.between(lo, hi)) & papers.known_first]
    est, a, b = boot_share(g.ssa_first); rows.append((f"{lo}–{hi}", len(g), f"{est:.1%}", f"{a:.1%}–{b:.1%}"))
for y in (2024, 2025):
    g = papers[(papers.year == y) & papers.known_first & papers.any_ssa]
    est, a, b = boot_share(g.ssa_first); rows.append((f"{y}, papers with an SSA author", len(g), f"{est:.1%}", f"{a:.1%}–{b:.1%}"))
bs = pd.DataFrame(rows, columns=["period", "n", "SSA first author", "95% bootstrap interval"]); bs.to_csv(LOGS/"bootstrap_ssa_first.csv", index=False); bs

,period,n,SSA first author,95% bootstrap interval
0,2000–2009,2423,8.1%,7.0%–9.2%
1,2010–2019,3704,10.1%,9.2%–11.1%
2,2020–2026,3060,20.0%,18.5%–21.4%
3,"2024, papers with an SSA author",146,61.0%,52.7%–68.5%
4,"2025, papers with an SSA author",200,67.0%,60.5%–73.5%


## Counts per year (and the incomplete last year)

In [5]:
yr = papers.groupby("year").size(); print(yr.loc[2015:2026].to_string())

year
2015.0    444
2016.0    413
2017.0    439
2018.0    385
2019.0    411
2020.0    429
2021.0    486
2022.0    513
2023.0    432
2024.0    464
2025.0    488
2026.0    437


## Citation impact: FWCI, with a warning about skew
OpenAlex's field-weighted citation impact (FWCI) compares a paper's citations with similar papers
(field, year, type). We use papers from 2015–2022 so that each has had time to be cited.

In [6]:
oe_file = FIN / "openalex_enrich.parquet"
if oe_file.exists():
    oe = pd.read_parquet(oe_file)[["pmid", "fwci"]].dropna()
    pco = pd.read_parquet(FIN / "paper_country_openalex.parquet")
    d = pco[pco.year.between(2015, 2022)].merge(oe, on="pmid")
    print(f"papers with FWCI 2015–2022: {d.pmid.nunique():,}; overall mean {oe.fwci.mean():.2f}, median {oe.fwci.median():.2f}")
    rows = []
    for c in ["USA", "GBR", "AUS", "DEU", "FRA", "IND", "CHN", "KEN", "MLI", "BFA", "GHA", "TZA"]:
        g = d[d.country == c].drop_duplicates("pmid")
        bs = rng.choice(g.fwci.to_numpy(), (2000, len(g))).mean(1)
        rows.append((c, len(g), round(g.fwci.mean(), 2), f"{np.percentile(bs, 2.5):.2f}–{np.percentile(bs, 97.5):.2f}", round(g.fwci.median(), 2)))
    fw = pd.DataFrame(rows, columns=["country", "papers", "mean FWCI", "95% bootstrap", "median FWCI"]).sort_values("mean FWCI", ascending=False)
    fw.to_csv(LOGS / "fwci_by_country.csv", index=False); print(fw.to_string(index=False))
    top = d.drop_duplicates("pmid").nlargest(5, "fwci")[["pmid", "fwci"]]
    print("Share of total FWCI from the top 1% of papers:",
          round(d.drop_duplicates("pmid").fwci.nlargest(int(0.01 * d.pmid.nunique())).sum() / d.drop_duplicates("pmid").fwci.sum(), 3))

papers with FWCI 2015–2022: 3,116; overall mean 4.31, median 1.82
country  papers  mean FWCI 95% bootstrap  median FWCI
    BFA      66       8.40    5.57–11.80         4.24
    GBR     597       5.97     4.75–7.44         2.43
    TZA      92       5.91     4.53–7.64         3.04
    KEN     150       4.99     3.69–6.57         2.28
    USA    1304       4.51     3.94–5.19         2.06
    DEU     243       4.51     3.66–5.45         2.07
    MLI      81       4.18     3.00–5.61         2.52
    FRA     225       4.05     3.23–4.95         1.92
    GHA     131       3.64     2.87–4.60         2.11
    CHN     156       3.58     1.64–7.17         1.18
    AUS     327       3.55     3.01–4.14         2.05
    IND     242       2.67     1.96–3.62         1.21
Share of total FWCI from the top 1% of papers: 0.186
